# Student Performance Prediction - walkthrough

Ovaj notebook je pomoćni materijal za razumevanje projekta, pripremu odbrane i kasnije pisanje formalne dokumentacije. Ne predstavlja glavni fajl za predaju i ne uvodi nove modele, treniranje, train/test split ili grafike.

## 1. Kratak opis projekta

Cilj projekta je predikcija završne ocene učenika, odnosno promenljive **G3**, na osnovu dostupnih atributa iz dataset-a `student-por.csv`.

U ovom trenutku projekat se nalazi u fazi pripreme podataka. To znači da se podaci učitavaju, proveravaju i organizuju za kasnije modelovanje, ali se model još ne trenira.

## 2. Opis dataset-a

Dataset se nalazi na putanji `data/raw/student-por.csv`.

Podaci potiču iz istraživanja školskog uspeha učenika u Portugalu. Dataset sadrži demografske, porodične, školske i ponašajne atribute učenika, kao i njihove ocene.

Važno je zapamtiti da su ocene u ovom dataset-u na skali **0-20**, a ne na skali 1-5.

- **G1** je ocena iz prvog perioda.
- **G2** je ocena iz drugog perioda.
- **G3** je završna ocena i ciljna promenljiva koju želimo da predvidimo.

In [ ]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "student-por.csv"
LOGS_DIR = PROJECT_ROOT / "data" / "logs"

In [ ]:
df = pd.read_csv(DATA_PATH)
df.head()

## 3. Zašto je ovo regresioni problem?

Ovo je regresioni problem zato što je ciljna promenljiva **G3 numerička vrednost**. Model treba da predvidi brojčanu završnu ocenu učenika na skali 0-20.

Kod klasifikacije bismo predviđali klasu ili kategoriju, na primer `položio` / `nije položio`. Ovde ne predviđamo kategoriju, već konkretnu numeričku ocenu, pa je zadatak regresija.

## 4. Zašto posebno analiziramo G1 i G2?

**G1** i **G2** su prethodne ocene učenika. Pošto predstavljaju raniji uspeh istog učenika, očekivano je da budu jako povezane sa završnom ocenom **G3**.

Zbog toga će model koji koristi G1 i G2 verovatno imati bolje rezultate. Međutim, model bez G1 i G2 može biti praktično korisniji, jer omogućava raniju procenu uspeha učenika, pre nego što su prethodne periodične ocene poznate ili kada želimo da se oslonimo samo na druge karakteristike.

## 5. Planirani scenariji modelovanja

U projektu su trenutno pripremljena dva glavna scenarija:

- **with_G1_G2**: koriste se svi ulazni atributi osim ciljne promenljive G3.
- **without_G1_G2**: koriste se svi ulazni atributi osim G1, G2 i G3.

Postoji i plan za treći scenario, **top_features**, ali on se još ne implementira. Biće definisan kasnije, nakon treniranja modela i analize važnosti atributa, odnosno feature importance analize.

## 6. Šta radi `step01_data_preparation.py`?

Fajl `src/step01_data_preparation.py` objedinjuje prvi korak pripreme podataka. On:

- učitava dataset,
- proverava missing vrednosti,
- proverava duplikate,
- proverava osnovne anomalije,
- proverava da li su G1, G2 i G3 u očekivanom opsegu 0-20,
- definiše ciljnu promenljivu G3,
- definiše scenarije `with_G1_G2` i `without_G1_G2`,
- priprema preprocessing logiku,
- generiše CSV izveštaje u folderu `data/logs/`.

U ovom koraku se ne trenira model, ne pravi se train/test split, ne fituje se `ColumnTransformer` i ne prave se grafici.

## 7. CSV izveštaji

Skripta `step01_data_preparation.py` generiše tri CSV izveštaja:

- `data/logs/dataset_overview.csv`
- `data/logs/features_overview.csv`
- `data/logs/preprocessing_report.csv`

Ovi fajlovi služe da se rezultati prvog koraka ne objašnjavaju samo kroz kod, već i kroz čitljive izveštaje koji se mogu koristiti tokom odbrane i pri pisanju dokumentacije.

### 7.1 `dataset_overview.csv`

Ovaj izveštaj daje osnovni pregled dataset-a: broj redova i kolona, broj duplikata, broj nedostajućih vrednosti, broj numeričkih i kategorijskih kolona, kao i osnovne informacije o ocenama G1, G2 i G3.

In [ ]:
dataset_overview = pd.read_csv(LOGS_DIR / "dataset_overview.csv")
dataset_overview

### 7.2 `features_overview.csv`

Ovaj izveštaj opisuje svaku kolonu u dataset-u. Za svaku kolonu prikazuje tip podatka, da li je numerička, kategorijska ili ciljna promenljiva, broj missing vrednosti, broj jedinstvenih vrednosti i da li se kolona koristi u scenarijima sa G1 i G2 ili bez G1 i G2.

Posebno je važno primetiti da se **G3** ne koristi kao ulazni atribut, jer je to ciljna promenljiva.

In [ ]:
features_overview = pd.read_csv(LOGS_DIR / "features_overview.csv")
features_overview

### 7.3 `preprocessing_report.csv`

Ovaj izveštaj sažima informacije vezane za početni preprocessing: missing vrednosti, duplikate, opseg ocena, anomalije u starosti, maksimalan broj izostanaka, ciljnu promenljivu, scenarije modelovanja, planirani preprocessing i status `top_features` scenarija.

In [ ]:
preprocessing_report = pd.read_csv(LOGS_DIR / "preprocessing_report.csv")
preprocessing_report

## 8. Šta znači numeric i categorical feature?

**Numeric feature** je atribut koji je predstavljen brojevima. Primeri su `age`, `absences`, `G1`, `G2` i slične kolone.

**Categorical feature** je atribut koji predstavlja kategoriju ili tekstualnu vrednost. Primeri su `school`, `sex`, `address`, `Mjob`, `Fjob` i slične kolone.

Ova podela je važna zato što se numerički i kategorijski atributi ne pripremaju na isti način. Numerički atributi se kasnije skaliraju, dok se kategorijski atributi kasnije enkodiraju.

## 9. Šta znači StandardScaler + OneHotEncoder?

`StandardScaler` se koristi za numeričke kolone. Njegova uloga je da numeričke vrednosti dovede na uporedivu skalu, što je korisno za mnoge algoritme mašinskog učenja.

`OneHotEncoder` se koristi za kategorijske kolone. On tekstualne kategorije pretvara u numerički oblik koji model može da koristi.

`ColumnTransformer` omogućava da se različite grupe kolona obrade različitim metodama. U ovom projektu to znači da se numeričke kolone obrađuju pomoću `StandardScaler`, a kategorijske kolone pomoću `OneHotEncoder(handle_unknown="ignore")`.

U ovom koraku se preprocessing samo priprema. To znači da objekti postoje u kodu, ali se ne poziva `.fit()`, ne pravi se train/test split i ne trenira se model.

# Step 02 - Eksplorativna analiza podataka

U drugom koraku projekta radi se eksplorativna analiza podataka kroz fajl `src/step02_eda.py`. Taj korak služi da razumemo raspodelu završne ocene, da proverimo korelacije između numeričkih atributa i da uočimo anomalije ili ekstremne vrednosti koje mogu biti važne za kasnije modelovanje.

Ovaj korak pokriva tri stvari: analizu korelacija, analizu anomalija i pregled ekstremnih vrednosti. Zbog toga gledamo G3 distribuciju, korelacionu matricu, scatter plotove i raspodelu izostanaka.

Grafici koji se nalaze u `data/graphs/` su:

- `g3_distribution.png`
- `numeric_correlation_matrix.png`
- `g1_vs_g3.png`
- `g2_vs_g3.png`
- `failures_vs_g3.png`
- `studytime_vs_g3.png`
- `absences_distribution.png`

Korelacija pokazuje koliko su dve numeričke promenljive povezane. Vrednost blizu 1 znači jaku pozitivnu vezu, vrednost blizu -1 znači jaku negativnu vezu, a vrednost blizu 0 znači da veza nije jaka ili nije linearna. U našem projektu je posebno važno da vidimo koliko su G1 i G2 povezani sa G3.

Fajl `data/logs/eda_report.csv` sadrži sažetak najvažnijih nalaza: broj učenika, opseg i prosečnu vrednost G3, broj učenika sa G3 = 0, ključne korelacije, najjaču korelaciju sa G3 i zaključak iz EDA analize.

G1 i G2 su bitni jer predstavljaju prethodne ocene i očekivano su najjače povezani sa završnom ocenom G3. Zbog toga je smisleno kasnije porediti modele sa i bez ovih atributa.

G3 = 0 i veće vrednosti `absences` se za sada ne brišu. U EDA fazi mi ih samo evidentiramo, jer mogu predstavljati realne slučajeve u dataset-u, a odluka o daljoj obradi dolazi kasnije.

Kratko objašnjenje najvažnijih delova `step02_eda.py`:

- prvo se učitava dataset iz `data/raw/student-por.csv`,
- zatim se računa korelaciona matrica za numeričke kolone,
- iz te matrice se pravi `numeric_correlation_matrix.png`,
- zatim se prave grafici za G1/G3, G2/G3, failures, studytime i absences,
- na kraju se pravi `eda_report.csv` sa glavnim zaključcima.

### Kako bih ovaj EDA korak objasnio na odbrani?

U drugom koraku sam uradio eksplorativnu analizu podataka da bih razumeo kako su atributi povezani sa završnom ocenom G3. Posebno sam gledao korelacije između prethodnih ocena G1 i G2 i ciljne promenljive, jer one najviše utiču na kasnije modelovanje. Takođe sam proverio raspodelu ocene G3, broj izostanaka i slučajeve sa G3 = 0. Na osnovu ovih nalaza nisam brisao te vrednosti, jer mogu biti stvarni i važni za model. Ovaj korak mi je pomogao da vidim koji atributi nose najviše informacija i da kasnije smisleno poredim modele sa i bez G1 i G2.


## EDA izveštaj

Ova ćelija samo učitava postojeći `eda_report.csv` i prikazuje ga, bez pravljenja novih statistika.

In [ ]:
eda_report = pd.read_csv(LOGS_DIR / "eda_report.csv")
eda_report

## EDA Grafici

Sledeće ćelije samo prikazuju već sačuvane PNG fajlove iz `data/graphs/`. Ne generiše se ništa novo.

In [ ]:
from IPython.display import Image, display

graph_paths = [
    ("Raspodela G3", PROJECT_ROOT / "data" / "graphs" / "g3_distribution.png"),
    ("Korelaciona matrica", PROJECT_ROOT / "data" / "graphs" / "numeric_correlation_matrix.png"),
    ("G1 vs G3", PROJECT_ROOT / "data" / "graphs" / "g1_vs_g3.png"),
    ("G2 vs G3", PROJECT_ROOT / "data" / "graphs" / "g2_vs_g3.png"),
    ("failures vs G3", PROJECT_ROOT / "data" / "graphs" / "failures_vs_g3.png"),
    ("studytime vs G3", PROJECT_ROOT / "data" / "graphs" / "studytime_vs_g3.png"),
    ("absences raspodela", PROJECT_ROOT / "data" / "graphs" / "absences_distribution.png"),
]

for title, graph_path in graph_paths:
    print(title)
    display(Image(filename=str(graph_path)))


## Objašnjenje najvažnijih delova koda iz `step01_data_preparation.py`

Ova sekcija ne kopira ceo Python fajl, nego izdvaja delove koje je najvažnije razumeti za odbranu. Cilj je da možemo jasno da objasnimo šta prvi korak projekta radi i zašto je urađen baš na taj način.

### 1. Definisanje putanja pomoću `pathlib Path`

U skripti se putanje definišu pomoću `Path`, na primer:

```python
PROJECT_ROOT = Path(__file__).resolve().parents[1]
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "student-por.csv"
LOGS_DIR = PROJECT_ROOT / "data" / "logs"
```

Ideja je da se prvo pronađe koren projekta, a zatim da se na njega nadovežu relativne putanje do dataset-a i log foldera. U nekim objašnjenjima se za koren projekta koristi naziv `BASE_DIR`; u ovom fajlu se ista ideja zove `PROJECT_ROOT`.

Ovo je bolje nego ručno pisanje apsolutnih putanja, jer apsolutna putanja važi samo na jednom računaru. Ako projekat prebacimo na drugi računar, `Path` pristup i dalje radi dokle god je struktura foldera ista.

### 2. Učitavanje dataset-a

Dataset se učitava pomoću pandas biblioteke:

```python
dataset = pd.read_csv(DATA_PATH)
```

U samoj skripti promenljiva se zove `df`, ali značenje je isto: učitavaju se podaci iz `data/raw/student-por.csv` u tabelarni oblik, da bi mogli da se analiziraju kolonama i redovima.

### 3. Funkcija `save_csv_if_changed`

Ova funkcija služi da se CSV fajlovi ne prepisuju bez potrebe. Skraćena logika izgleda ovako:

```python
if not path.exists():
    new_dataframe.to_csv(path, index=False)
elif existing_dataframe.equals(new_dataframe_for_compare):
    print("CSV fajl nije promenjen")
else:
    new_dataframe.to_csv(path, index=False)
```

Ako CSV ne postoji, funkcija ga kreira. Ako CSV postoji i sadržaj je isti, fajl se ne menja. Ako CSV postoji, ali se sadržaj razlikuje, fajl se ažurira.

Ovo je korisno zato što `git status` ostaje čist kada nema stvarnih promena. Drugim rečima, skripta ne pravi lažne izmene samo zato što je ponovo pokrenuta.

### 4. Pravljenje `dataset_overview.csv`

`dataset_overview.csv` je ljudski čitljiv pregled dataset-a. On ne služi za treniranje modela, nego za brzo razumevanje osnovnog stanja podataka.

U njemu se nalaze informacije kao što su broj redova, broj kolona, broj missing vrednosti, broj duplikata i osnovne statistike za ocene G1, G2 i G3. Na odbrani se ovaj fajl može koristiti da se pokaže da je dataset pregledan pre bilo kakvog modelovanja.

### 5. Pravljenje `features_overview.csv`

`features_overview.csv` objedinjuje pregled svih atributa. Za svaku kolonu prikazuje naziv kolone, tip podatka, da li je kolona `numeric`, `categorical` ili `target`, broj missing vrednosti, broj jedinstvenih vrednosti i način korišćenja u scenarijima.

Posebno je važno sledeće:

- **G3** je target i ne koristi se kao ulazni atribut.
- **G1** i **G2** su numerički atributi i koriste se samo u scenariju `with_G1_G2`.
- Ostali atributi se koriste u oba scenarija.

Ovaj CSV je praktičan jer na jednom mestu pokazuje i tehnički tip kolone i njenu ulogu u projektu.

### 6. Definisanje scenarija

Dva scenarija se mogu objasniti ovim pseudokodom:

```python
features_with_g1_g2 = sve kolone osim G3
features_without_g1_g2 = sve kolone osim G1, G2 i G3
```

Prvi scenario koristi prethodne ocene G1 i G2, pa očekujemo da će kasnije dati bolje rezultate. Drugi scenario ih izbacuje, jer želimo da proverimo koliko model može biti koristan kada se uspeh učenika procenjuje ranije ili bez oslanjanja na prethodne ocene.

### 7. Priprema preprocessing-a

Preprocessing se priprema tako što se numeričke i kategorijske kolone obrađuju različito:

```python
ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)
```

Numeričke kolone idu kroz `StandardScaler`, koji ih kasnije dovodi na uporedivu skalu. Kategorijske kolone idu kroz `OneHotEncoder`, koji tekstualne kategorije pretvara u numerički oblik. `ColumnTransformer` omogućava da se ove dve grupe kolona obrade različitim postupcima u istom preprocessing koraku.

U ovom koraku se preprocessing samo priprema. Ne poziva se `.fit()`, ne radi se train/test split i ne trenira se model.

### Kako bih ovo objasnio na odbrani?

U prvom koraku sam pripremio dataset za kasniju analizu i modelovanje. Prvo sam učitao podatke iz `student-por.csv` i proverio osnovne informacije, kao što su broj redova, broj kolona, missing vrednosti i duplikati. Zatim sam proverio da li su ocene G1, G2 i G3 u očekivanom opsegu od 0 do 20. Posebno sam izdvojio G3 kao ciljnu promenljivu, jer nju želimo da predvidimo. Definisao sam dva scenarija: jedan u kome koristimo G1 i G2 i drugi u kome ih ne koristimo. Na kraju sam pripremio preprocessing logiku, gde će se numeričke kolone skalirati, a kategorijske enkodirati. U ovom koraku još nema treniranja modela, već se samo proverava i organizuje dataset za sledeće faze projekta.


## 10. Kratak zaključak posle prvog koraka

Posle prvog koraka dataset je učitan i pregledan. Na osnovu CSV izveštaja može se proveriti da li postoje nedostajuće vrednosti, duplikati, anomalije i kolone koje bi bile očigledni kandidati za uklanjanje.

Prema trenutnom `preprocessing_report.csv`, dataset nema nedostajuće vrednosti i duplikate, a G1, G2 i G3 su u očekivanom opsegu 0-20. Takođe, nema očiglednih ID ili konstantnih kolona za uklanjanje.

Sledeći korak projekta biće EDA analiza i grafici, kako bi se bolje razumele veze između atributa i ciljne promenljive G3.